# Speech Error Diagnostics and Governance Lab


## Purpose

This lab simulates speech recognition error diagnostics.

The central point: aggregate WER can hide uneven error burdens across groups and conditions.

In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(42)
n = 1000

data = pd.DataFrame({
    "speaker_group": rng.choice(["A", "B", "C"], size=n, p=[0.5, 0.3, 0.2]),
    "noise_condition": rng.choice(["clean", "moderate_noise", "high_noise"], size=n),
    "reference_words": rng.integers(5, 31, size=n),
})

noise_multiplier = data["noise_condition"].map({
    "clean": 0.05,
    "moderate_noise": 0.12,
    "high_noise": 0.22,
})

group_multiplier = data["speaker_group"].map({
    "A": 1.00,
    "B": 1.15,
    "C": 1.35,
})

expected_errors = data["reference_words"] * noise_multiplier * group_multiplier

data["substitutions"] = rng.poisson(expected_errors * 0.45)
data["deletions"] = rng.poisson(expected_errors * 0.35)
data["insertions"] = rng.poisson(expected_errors * 0.20)

data["wer"] = (
    data["substitutions"] +
    data["deletions"] +
    data["insertions"]
) / data["reference_words"]

summary = (
    data
    .groupby(["speaker_group", "noise_condition"], as_index=False)
    .agg(
        sample_size=("wer", "size"),
        mean_wer=("wer", "mean"),
        median_wer=("wer", "median"),
    )
)

summary

In [ ]:
summary.to_csv("../outputs/notebook_speech_error_diagnostics.csv", index=False)
summary

## Review Questions

- Which group-condition pair has the highest WER?
- Is the difference practically meaningful?
- Would this system be acceptable for accessibility use?
- What additional evaluation data would be required?
- Who should review deployment?